# EXP-011: PLR-MLP + Bin-cat (Dual Representation, MLP only)

**근거:** EXP-009/010에서 PLR이 진짜 효과 입증 (LB +0.00008, 100% 전이). RealMLP의 또 다른 기법인 **numeric→bin-cat dual representation**을 자체 구현해서 빠른 검증.

**Bin-cat 직접 구현 (RealMLP `feature_engineering` 패턴):**
1. 각 NUM_COL → `np.floor()` → train 전체로 `factorize()` (라벨 무관, 누수 없음)
2. test에 mapping 적용 (unseen → 0 unknown slot)
3. 11개 bin-cat 컬럼 추가, 각각 embedding dim=4 → 44 더 입력

**Dual representation:** 같은 numeric이 (1) raw BN (2) PLR encoder (3) bin-cat embedding 세 표현으로 모델에 입력.

**변경점 vs EXP-009 PLR-MLP:** bin-cat 추가만 (PLR + nothing else). MLP-only 학습.

In [1]:
import pandas as pd
import numpy as np
import time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
NUM_COLS = ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
            'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
            'RaceProgress', 'Position_Change']
y = train[TARGET].astype(int).values

# Driver/Compound/Race LE
train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

N_FOLDS, SEED = 5, 42
print(f'Train: {train.shape}, Test: {test.shape}')

Train: (439140, 16), Test: (188165, 15)


## 1. Bin-cat 직접 구현 (RealMLP feature_engineering 패턴)

- train의 numeric을 `np.floor`로 정수화 → factorize로 0..N-1 mapping
- test에 같은 mapping 적용. unseen은 -1 (나중에 0 unknown으로 offset)
- 라벨 무관 통계라 누수 없음

In [2]:
# np.floor → factorize on train, map to test
category_map = {}
bin_cat_tr = np.zeros((len(train), len(NUM_COLS)), dtype=np.int64)
bin_cat_te = np.zeros((len(test), len(NUM_COLS)), dtype=np.int64)
bin_cat_cards = []

for i, col in enumerate(NUM_COLS):
    floored_tr = np.floor(train[col].values).astype(np.int64)
    codes_tr, uniques = pd.factorize(floored_tr)
    category_map[col] = {v: idx for idx, v in enumerate(uniques)}
    bin_cat_tr[:, i] = codes_tr  # 0..N-1

    floored_te = np.floor(test[col].values).astype(np.int64)
    code_map = category_map[col]
    bin_cat_te[:, i] = np.array([code_map.get(v, -1) for v in floored_te])

    n_unique = len(uniques)
    n_unseen_te = int((bin_cat_te[:, i] == -1).sum())
    bin_cat_cards.append(n_unique + 1)  # +1 for unknown slot at 0
    print(f'  {col:25s} unique_train={n_unique:5d}, unseen_test={n_unseen_te:5d}')

# Offset to +1 (0 = unknown slot)
bin_cat_tr = bin_cat_tr + 1
bin_cat_te = bin_cat_te + 1
bin_cat_te[bin_cat_te < 0] = 0  # safety: -1+1=0 already, but for any -1 leftover

print(f'\nbin_cat cardinalities (with unknown): {bin_cat_cards}')
print(f'bin_cat_tr shape: {bin_cat_tr.shape}, bin_cat_te shape: {bin_cat_te.shape}')

  Year                      unique_train=    4, unseen_test=    0
  PitStop                   unique_train=    2, unseen_test=    0
  LapNumber                 unique_train=   78, unseen_test=    0
  Stint                     unique_train=    8, unseen_test=    0
  TyreLife                  unique_train=   77, unseen_test=    0
  Position                  unique_train=   20, unseen_test=    0
  LapTime (s)               unique_train=  132, unseen_test=    3
  LapTime_Delta             unique_train=  218, unseen_test=   13
  Cumulative_Degradation    unique_train=  445, unseen_test=    8
  RaceProgress              unique_train=    2, unseen_test=    0
  Position_Change           unique_train=   37, unseen_test=    0

bin_cat cardinalities (with unknown): [5, 3, 79, 9, 78, 21, 133, 219, 446, 3, 38]
bin_cat_tr shape: (439140, 11), bin_cat_te shape: (188165, 11)


## 2. Cat / Num 데이터 준비

In [3]:
drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]

# Original cat → indexing
cat_cards = []
X_cat_tr = np.zeros((len(X_le), len(CAT_COLS)), dtype=np.int64)
X_cat_te = np.zeros((len(X_test_le), len(CAT_COLS)), dtype=np.int64)
for i, c in enumerate(CAT_COLS):
    n_unique = X_le[c].max() + 1
    cat_cards.append(n_unique + 1)
    X_cat_tr[:, i] = X_le[c].values + 1
    te_vals = X_test_le[c].values + 1
    te_vals[te_vals < 0] = 0
    X_cat_te[:, i] = te_vals

X_num_tr_raw = X_le[NUM_COLS].values.astype(np.float32)
X_num_te_raw = X_test_le[NUM_COLS].values.astype(np.float32)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X_le, y))

## 3. PLR-MLP + Bin-cat 모델

- cat_emb (28) + bin_cat_emb (11 × 4 = 44) + raw_num_BN (11) + PLR (88) = **171-dim input**
- Body: `[171] → 256 → 128 → 64 → 1`

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

class PLREncoder(nn.Module):
    def __init__(self, n_numeric, k=16, hidden=8, sigma=2.33):
        super().__init__()
        self.freqs = nn.Parameter(torch.randn(n_numeric, k) * sigma)
        self.linear = nn.Linear(2 * k, hidden)
        self.act = nn.GELU()
    def forward(self, x_num):
        z = 2 * math.pi * x_num.unsqueeze(-1) * self.freqs.unsqueeze(0)
        p = torch.cat([torch.sin(z), torch.cos(z)], dim=-1)
        out = self.act(self.linear(p))
        return out.flatten(start_dim=1)


class PLRBinCatMLP(nn.Module):
    def __init__(self, cat_cards, emb_dims, bincat_cards, bincat_emb_dim,
                 n_numeric, plr_k=16, plr_hidden=8,
                 hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        # Original cat embeddings
        self.embs = nn.ModuleList([nn.Embedding(card, dim) for card, dim in zip(cat_cards, emb_dims)])
        emb_total = sum(emb_dims)
        # Bin-cat embeddings (same dim for all)
        self.bincat_embs = nn.ModuleList([nn.Embedding(card, bincat_emb_dim) for card in bincat_cards])
        bincat_total = len(bincat_cards) * bincat_emb_dim
        # Raw numeric BN
        self.bn_num = nn.BatchNorm1d(n_numeric)
        # PLR
        self.plr = PLREncoder(n_numeric, k=plr_k, hidden=plr_hidden, sigma=2.33)
        plr_total = n_numeric * plr_hidden
        # Body
        in_dim = emb_total + bincat_total + n_numeric + plr_total
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.body = nn.Sequential(*layers)
        self.in_dim = in_dim
        print(f'PLRBinCatMLP input dim: cat_emb={emb_total} + bin_cat_emb={bincat_total} + raw_num={n_numeric} + plr={plr_total} = {in_dim}')

    def forward(self, x_cat, x_bin_cat, x_num):
        emb = torch.cat([emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embs)], dim=1)
        bincat_emb = torch.cat([emb_layer(x_bin_cat[:, i]) for i, emb_layer in enumerate(self.bincat_embs)], dim=1)
        raw_num = self.bn_num(x_num)
        plr_feat = self.plr(x_num)
        x = torch.cat([emb, bincat_emb, raw_num, plr_feat], dim=1)
        return self.body(x).squeeze(1)

device: cuda


## 4. 5-fold 학습

In [5]:
EMB_DIMS = [16, 4, 8]
BINCAT_EMB_DIM = 4
BATCH_SIZE = 4096
MAX_EPOCHS = 80
PATIENCE = 20

oof_mlp = np.zeros(len(X_le))
test_mlp = np.zeros(len(X_test_le))
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    scaler = StandardScaler()
    X_num_tr = scaler.fit_transform(X_num_tr_raw[tr_idx])
    X_num_va = scaler.transform(X_num_tr_raw[va_idx])
    X_num_te = scaler.transform(X_num_te_raw)

    Xc_tr = torch.tensor(X_cat_tr[tr_idx], dtype=torch.long)
    Xb_tr = torch.tensor(bin_cat_tr[tr_idx], dtype=torch.long)
    Xn_tr = torch.tensor(X_num_tr, dtype=torch.float32)
    yt_tr = torch.tensor(y[tr_idx], dtype=torch.float32)
    Xc_va = torch.tensor(X_cat_tr[va_idx], dtype=torch.long, device=device)
    Xb_va = torch.tensor(bin_cat_tr[va_idx], dtype=torch.long, device=device)
    Xn_va = torch.tensor(X_num_va, dtype=torch.float32, device=device)
    yt_va = y[va_idx]
    Xc_te = torch.tensor(X_cat_te, dtype=torch.long, device=device)
    Xb_te = torch.tensor(bin_cat_te, dtype=torch.long, device=device)
    Xn_te = torch.tensor(X_num_te, dtype=torch.float32, device=device)

    train_loader = DataLoader(
        TensorDataset(Xc_tr, Xb_tr, Xn_tr, yt_tr),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        num_workers=0, pin_memory=True,
    )

    torch.manual_seed(SEED)
    model = PLRBinCatMLP(cat_cards, EMB_DIMS, bin_cat_cards, BINCAT_EMB_DIM, len(NUM_COLS)).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=MAX_EPOCHS)
    bce = nn.BCEWithLogitsLoss()

    best_auc = -1
    best_state = None
    wait = 0
    epochs_used = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xc, xb, xn, yb in train_loader:
            xc = xc.to(device, non_blocking=True)
            xb = xb.to(device, non_blocking=True)
            xn = xn.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optim.zero_grad()
            logit = model(xc, xb, xn)
            loss = bce(logit, yb)
            loss.backward()
            optim.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            va_logit = model(Xc_va, Xb_va, Xn_va).cpu().numpy()
        va_prob = 1 / (1 + np.exp(-va_logit))
        auc = roc_auc_score(yt_va, va_prob)
        epochs_used = epoch + 1

        if auc > best_auc + 1e-6:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        va_logit = model(Xc_va, Xb_va, Xn_va).cpu().numpy()
        te_logit = model(Xc_te, Xb_te, Xn_te).cpu().numpy()
    oof_mlp[va_idx] = 1 / (1 + np.exp(-va_logit))
    test_mlp += (1 / (1 + np.exp(-te_logit))) / N_FOLDS
    early_stopped = epochs_used < MAX_EPOCHS
    print(f'PLR+BinCat MLP fold {fold}: best AUC={best_auc:.5f}, epochs_used={epochs_used} {"(ES)" if early_stopped else "(max)"}')

    del model, optim, sched, Xc_va, Xb_va, Xn_va, Xc_te, Xb_te, Xn_te
    torch.cuda.empty_cache()

mlp_oof_auc = roc_auc_score(y, oof_mlp)
print(f'\nPLR+BinCat MLP OOF AUC: {mlp_oof_auc:.5f}')
print(f'  vs EXP-009 PLR-MLP    0.94486: {mlp_oof_auc-0.94486:+.5f}')
print(f'  vs EXP-008 MLP        0.94095: {mlp_oof_auc-0.94095:+.5f}')
print(f'  elapsed: {time.time()-t0:.1f}s')

PLRBinCatMLP input dim: cat_emb=28 + bin_cat_emb=44 + raw_num=11 + plr=88 = 171
PLR+BinCat MLP fold 0: best AUC=0.94583, epochs_used=32 (ES)
PLRBinCatMLP input dim: cat_emb=28 + bin_cat_emb=44 + raw_num=11 + plr=88 = 171
PLR+BinCat MLP fold 1: best AUC=0.94349, epochs_used=31 (ES)
PLRBinCatMLP input dim: cat_emb=28 + bin_cat_emb=44 + raw_num=11 + plr=88 = 171
PLR+BinCat MLP fold 2: best AUC=0.94473, epochs_used=33 (ES)
PLRBinCatMLP input dim: cat_emb=28 + bin_cat_emb=44 + raw_num=11 + plr=88 = 171
PLR+BinCat MLP fold 3: best AUC=0.94384, epochs_used=38 (ES)
PLRBinCatMLP input dim: cat_emb=28 + bin_cat_emb=44 + raw_num=11 + plr=88 = 171
PLR+BinCat MLP fold 4: best AUC=0.94470, epochs_used=31 (ES)

PLR+BinCat MLP OOF AUC: 0.94444
  vs EXP-009 PLR-MLP    0.94486: -0.00042
  vs EXP-008 MLP        0.94095: +0.00349
  elapsed: 705.8s


## 5. Save predictions + 결론

In [6]:
np.save('../submissions/exp011_plr_bincat_mlp_oof.npy', oof_mlp)
np.save('../submissions/exp011_plr_bincat_mlp_test.npy', test_mlp)
sub = pd.DataFrame({'id': test['id'], 'PitNextLap': test_mlp})
sub.to_csv('../submissions/submission_exp011_plr_bincat_mlp.csv', index=False)
print(f'saved exp011 OOF/test arrays + submission CSV')
print(f'test pred mean: {test_mlp.mean():.4f}')
print()
print('=== EXP-011 결론 ===')
print(f'  EXP-009 PLR-MLP only       : 0.94486')
print(f'  EXP-011 PLR+BinCat MLP only: {mlp_oof_auc:.5f}')
print(f'  Δ (bin-cat 효과)            : {mlp_oof_auc-0.94486:+.5f}')
print()
delta = mlp_oof_auc - 0.94486
if delta > 0.002:
    print(f'  → Bin-cat 효과 큼. EXP-012에서 4-way 풀 학습 진행')
elif delta > 0.0005:
    print(f'  → Bin-cat 효과 중간. EXP-012 4-way 학습 가치')
elif delta > -0.0005:
    print(f'  → Bin-cat 효과 미미. 다른 기법 (inner ens, robust scale) 우선순위')
else:
    print(f'  → Bin-cat 부정. 본 데이터에 dual representation 부적합')

saved exp011 OOF/test arrays + submission CSV
test pred mean: 0.2025

=== EXP-011 결론 ===
  EXP-009 PLR-MLP only       : 0.94486
  EXP-011 PLR+BinCat MLP only: 0.94444
  Δ (bin-cat 효과)            : -0.00042

  → Bin-cat 효과 미미. 다른 기법 (inner ens, robust scale) 우선순위
